# Data exploration nach CRISP-DM

In [1]:
#!/usr/bin/env python3

# Bibliotheken laden
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm


# Pfade definieren
price_path   = Path("/Volumes/scratch-apfs-owc-1/PredAna_Python/tankerkoenig-data/prices/")
station_path = Path("/Volumes/scratch-apfs-owc-1/PredAna_Python/tankerkoenig-data/stations/")

# Stichprobe einer einzelnen Datei laden
price_path_example = price_path / "2024/04/2024-04-01-prices.csv"
df_prices   = pd.read_csv(price_path_example)
df_stations = pd.read_csv(station_path / "stations.csv")

print("Übersicht Preise:\n")
display(df_prices.head())
print("\nÜbersicht Stationen:\n")
display(df_stations.head())

print("\nStatistiken über df_prices:\n")
display(df_prices.describe())
print("\nStatistiken über df_stations:\n")
display(df_stations.describe())

print("Fehlende Werte in df_prices:\n" + str(df_prices.isnull().sum()))
print("\nFehlende Werte in df_stations:\n" + str(df_stations.isnull().sum()))

print("\nDatentypen in df_prices:\n"   + str(df_prices.dtypes))
print("\nDatentypen in df_stations:\n" + str(df_stations.dtypes))

Übersicht Preise:



,date,station_uuid,diesel,e5,e10,dieselchange,e5change,e10change
0,2024-04-01 00:00:34+02,db307731-f6c4-4c45-9af4-1058e9b23397,1.709,1.839,1.779,1,1,1
1,2024-04-01 00:00:34+02,6bb23c12-f7ce-41a8-ad1c-b50854808fe9,1.699,1.779,1.719,1,1,1
2,2024-04-01 00:00:34+02,c7f687c8-5718-49d0-88b0-092c55aeb9a4,1.689,1.879,1.819,1,1,1
3,2024-04-01 00:00:34+02,c1a1a169-d0b9-4e6a-a9a0-a44c4065e8f7,1.829,1.959,1.899,1,1,1
4,2024-04-01 00:00:34+02,a9994385-741f-4374-b738-d33681e101a7,1.699,1.879,1.819,1,0,0



Übersicht Stationen:



,uuid,name,brand,street,house_number,post_code,city,latitude,longitude
0,00060723-0001-4444-8888-acdc00000001,BAGeno Raiffeisen eG,NaN,Künzelsauer Strasse,7,74653,Ingelfingen,49.296822,9.661385
1,005056ba-7cb6-1ed2-bceb-5332ab168d12,famila Tankstelle,FAMILA,Pascalstrasse,9,25442,Quickborn,53.742150,9.941240
2,005056ba-7cb6-1ed2-bceb-573c18314d16,star Tankstelle,STAR,Riehler Strasse,240,50735,Köln,50.961800,6.980070
3,005056ba-7cb6-1ed2-bceb-662ba1a94d1f,star Tankstelle,STAR,BAB 10 / Seeberg Ost,NaN,15345,Altlandsberg,52.550160,13.682120
4,005056ba-7cb6-1ed2-bceb-6f7b23564d23,star Tankstelle,STAR,Duisburger Straße,130,47166,Duisburg,51.489790,6.783730



Statistiken über df_prices:



,diesel,e5,e10,dieselchange,e5change,e10change
count,363053.000000,363053.000000,363053.000000,363053.000000,363053.000000,363053.000000
mean,1.726644,1.858053,1.751622,0.786871,0.743878,0.723886
std,0.060233,0.237702,0.368377,0.411176,0.437978,0.448397
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.689000,1.849000,1.789000,1.000000,0.000000,0.000000
50%,1.719000,1.879000,1.819000,1.000000,1.000000,1.000000
75%,1.759000,1.919000,1.859000,1.000000,1.000000,1.000000
max,2.259000,4.000000,2.399000,3.000000,3.000000,3.000000



Statistiken über df_stations:



,latitude,longitude
count,15442.000000,15442.000000
mean,814.298315,107.406623
std,6212.821365,1011.925374
min,0.000000,0.000000
25%,49.405422,8.036261
50%,50.995352,9.325267
75%,52.217675,11.115400
max,54813.000000,14883.000000


Fehlende Werte in df_prices:
date            0
station_uuid    0
diesel          0
e5              0
e10             0
dieselchange    0
e5change        0
e10change       0
dtype: int64

Fehlende Werte in df_stations:
uuid               0
name               0
brand            544
street             2
house_number    3678
post_code          2
city               3
latitude           0
longitude          0
dtype: int64

Datentypen in df_prices:
date             object
station_uuid     object
diesel          float64
e5              float64
e10             float64
dieselchange      int64
e5change          int64
e10change         int64
dtype: object

Datentypen in df_stations:
uuid             object
name             object
brand            object
street           object
house_number     object
post_code        object
city             object
latitude        float64
longitude       float64
dtype: object


In [2]:
# ── Konfiguration ──────────────────────────────────────────────────────────────

# Welche Spritsorten ausgewertet werden sollen
ACTIVE_FUELS = ["diesel", "e5", "e10"]

# Schwellwert ab dem ein Preis als Ausreißer gilt (je Spritsorte)
OUTLIER_THRESHOLDS = {"diesel": 3.0, "e5": 4.0, "e10": 4.0}

CHANGE_COLS = [f"{f}change" for f in ACTIVE_FUELS]
USECOLS     = ["date", "station_uuid"] + ACTIVE_FUELS + CHANGE_COLS
DTYPES      = {f: "float32" for f in ACTIVE_FUELS} | {f"{f}change": "int8" for f in ACTIVE_FUELS}


# ── Kombinierter Single-pass-Scan ───────────────────────────────────────────────

def scan_file(filepath: str) -> tuple[tuple, dict | None]:
    """
    Öffnet jede CSV einmal und berechnet dabei gleichzeitig:
      1) NaN/Zero-Statistiken  (früher: analyze_price_file)
      2) Ausreißer je Spritsorte  (früher: scan_file_all_fuels)
    """
    try:
        df = pd.read_csv(filepath, usecols=USECOLS, dtype=DTYPES)
    except Exception:
        return (0, 0, 0, 0, 0), None

    # 1) Statistiken
    nan_count    = int(df[ACTIVE_FUELS].isna().sum().sum())
    zero_count   = int((df[ACTIVE_FUELS] == 0).sum().sum())
    zero_removed = int(sum(
        ((df[fuel] == 0) & (df[f"{fuel}change"] == 2)).sum()
        for fuel in ACTIVE_FUELS
    ))
    stats = (len(df), nan_count, zero_count, zero_removed, zero_count - zero_removed)

    # 2) Ausreißer
    per_fuel = {}
    for fuel, target in OUTLIER_THRESHOLDS.items():
        mask  = np.isclose(df[fuel], target, atol=1e-6)
        count = int(mask.sum())
        if count > 0:
            hits = df.loc[mask, ["date", "station_uuid", fuel, f"{fuel}change"]].copy()
            hits.rename(columns={fuel: "price", f"{fuel}change": "change"}, inplace=True)
            hits["fuel"] = fuel
            hits["file"] = filepath
            per_fuel[fuel] = {"file": filepath, "fuel": fuel, "count": count, "rows": hits}

    return stats, (per_fuel if per_fuel else None)


# ── Executor ────────────────────────────────────────────────────────────────────

all_files = [str(p) for p in price_path.rglob("*.csv")]
print(f"Gefundene CSV-Dateien: {len(all_files)}\n")

total_rows = total_nan = total_zero = total_zero_removed = total_zero_not_removed = 0
results_by_fuel: dict[str, list] = {fuel: [] for fuel in OUTLIER_THRESHOLDS}
detail_parts: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=8) as executor:
    for stats, per_fuel in tqdm(
        executor.map(scan_file, all_files),
        total=len(all_files),
        desc="Single-pass Scan",
    ):
        row_count, nan_count, zero_count, zero_removed, zero_not_removed = stats
        total_rows            += row_count
        total_nan             += nan_count
        total_zero            += zero_count
        total_zero_removed    += zero_removed
        total_zero_not_removed += zero_not_removed

        if per_fuel:
            for fuel, result in per_fuel.items():
                results_by_fuel[fuel].append(result)

# ── Statistik-Ausgabe ───────────────────────────────────────────────────────────

print(f"\nGesamtzahl Zeilen:                  {total_rows:,}")
print(f"Echte NaN in Preisfeldern:          {total_nan:,}")
print(f"Preis == 0 in Preisfeldern:         {total_zero:,}")
print(f"Preis == 0 und change == 2:         {total_zero_removed:,}")
print(f"Preis == 0 und change != 2:         {total_zero_not_removed:,}")

# ── Ausreißer-Ausgabe ───────────────────────────────────────────────────────────

print()
for fuel, fuel_hits in results_by_fuel.items():
    if not fuel_hits:
        print(f"  {fuel}: kein Treffer")
    else:
        total_hits = sum(r["count"] for r in fuel_hits)
        print(f"  {fuel}: {total_hits:,} Treffer in {len(fuel_hits)} Dateien")
        detail_parts.append(pd.concat([r["rows"] for r in fuel_hits], ignore_index=True))

detail_df = (
    pd.concat(detail_parts, ignore_index=True)
    if detail_parts
    else pd.DataFrame(columns=["date", "station_uuid", "price", "change", "fuel", "file"])
)

for fuel, fuel_hits in results_by_fuel.items():
    if not fuel_hits:
        continue
    summary = (
        pd.DataFrame([{"file": r["file"], "count": r["count"]} for r in fuel_hits])
        .sort_values("count", ascending=False)
    )
    print(f"\nTop-10-Dateien für {fuel}:")
    display(summary.head(10))

Gefundene CSV-Dateien: 4365



Single-pass Scan:   0%|          | 0/4365 [00:00<?, ?it/s]


Gesamtzahl Zeilen:                  1,128,245,397
Echte NaN in Preisfeldern:          0
Preis == 0 in Preisfeldern:         61,521,119
Preis == 0 und change == 2:         831,021
Preis == 0 und change != 2:         60,690,098

  diesel: 193 Treffer in 97 Dateien
  e5: 21,976 Treffer in 833 Dateien
  e10: 13 Treffer in 9 Dateien

Top-10-Dateien für diesel:


,file,count
44,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,38
42,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,12
48,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,5
92,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,5
63,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,5
70,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,4
83,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,3
82,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,3
81,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,3
38,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,3



Top-10-Dateien für e5:


,file,count
321,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,56
318,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,55
334,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,52
285,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,51
342,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,51
333,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,50
327,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,50
320,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,47
336,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,47
316,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,46



Top-10-Dateien für e10:


,file,count
1,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,4
5,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,2
0,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
2,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
3,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
4,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
6,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
7,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1
8,/Volumes/scratch-apfs-owc-1/PredAna_Python/tan...,1


In [3]:
# Treffer pro Datei als CSV exportieren (je Spritsorte eine Datei)

for fuel, fuel_hits in results_by_fuel.items():
    if not fuel_hits:
        continue
    summary = (
        pd.DataFrame([{"count": r["count"], "filename": Path(r["file"]).name} for r in fuel_hits])
        .sort_values("count", ascending=False)
    )
    out_path = f"{fuel}_outlier_count_per_file.csv"
    summary.to_csv(out_path, index=False)
    print(f"Exportiert: {out_path}  ({len(summary)} Dateien)")

Exportiert: diesel_outlier_count_per_file.csv  (97 Dateien)
Exportiert: e5_outlier_count_per_file.csv  (833 Dateien)
Exportiert: e10_outlier_count_per_file.csv  (9 Dateien)


In [4]:
# Verdächtige UUIDs — gruppiert nach Spritsorte

suspect_df = (
    detail_df.groupby(["fuel", "station_uuid"])
    .size()
    .reset_index(name="outlier_count")
    .sort_values(["fuel", "outlier_count"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Verdächtige Station/Spritsorte-Kombinationen gesamt: {len(suspect_df)}\n")

for fuel in suspect_df["fuel"].unique():
    sub = suspect_df[suspect_df["fuel"] == fuel]
    print(f"{fuel} — {len(sub)} verdächtige Stationen:")
    display(sub.head(20))

Verdächtige Station/Spritsorte-Kombinationen gesamt: 36

diesel — 14 verdächtige Stationen:


,fuel,station_uuid,outlier_count
0,diesel,15884845-a2be-4f77-a37b-1ac9e876a62a,137
1,diesel,8099817a-75c0-42a6-8b84-cbbf456f41bb,30
2,diesel,88b513b9-1369-460c-b878-ab33e6b3a177,7
3,diesel,6363e06a-ff31-4938-bbda-7dbda4a796b4,5
4,diesel,49164e7b-24bc-424d-7122-eed448b4b80c,3
5,diesel,62d75b56-be9c-45b6-a263-20df7a78d7c6,2
6,diesel,aa842438-c80d-46c1-828f-2cadb756d011,2
7,diesel,047f303e-8978-4106-990d-e62b8c45c0d8,1
8,diesel,0ce2fe47-449c-4cfa-843e-818cd482b624,1
9,diesel,748223cd-6cb2-4619-b8bf-c4987821d0ae,1


e10 — 5 verdächtige Stationen:


,fuel,station_uuid,outlier_count
14,e10,44f1c1c2-621d-44ab-0dab-6dc7ff3b97fe,5
15,e10,49164e7b-24bc-424d-7122-eed448b4b80c,3
16,e10,618f05b4-1125-484a-b747-373882165ff1,3
17,e10,047f303e-8978-4106-990d-e62b8c45c0d8,1
18,e10,0f94fbcb-31ef-4fd9-9a45-7e25fc94b591,1


e5 — 17 verdächtige Stationen:


,fuel,station_uuid,outlier_count
19,e5,78dfc1c1-8e95-4ed2-bad0-14ecdb1e8163,17110
20,e5,b9b395af-04c3-4150-a81b-f4dd215a20f0,3044
21,e5,8e9ace8c-bd06-58a1-9ddc-e480e3b72b57,1339
22,e5,4be6cac8-e125-41eb-ac4f-4103f8735107,466
23,e5,8be7e249-180e-407a-8f3b-dcfae36c139c,2
24,e5,c1169472-dc4d-583d-81de-60440247d683,2
25,e5,d9879018-c48a-415e-a944-0c67c9618b36,2
26,e5,df7ab2a6-e897-44d5-8609-06d35592e7d6,2
27,e5,00099999-99e7-4444-8888-acdc00000076,1
28,e5,12bc699f-3a3f-4cd8-8970-2e41e04b5688,1


In [5]:
# Liste aller bekannten Stations-UUIDs aus stations.csv

station_files = list(station_path.rglob("*.csv"))
print(f"Gefundene Stationsdateien: {len(station_files)}")

def read_station_uuids(filepath):
    df = pd.read_csv(filepath, usecols=["uuid"], dtype={"uuid": "string"})
    return set(df["uuid"].dropna().unique())

known_station_ids = set()

with ThreadPoolExecutor(max_workers=8) as executor:
    for uuid_set in tqdm(
        executor.map(read_station_uuids, station_files),
        total=len(station_files),
        desc="Lese Stations-UUIDs"
    ):
        known_station_ids.update(uuid_set)

print(f"Anzahl eindeutiger UUIDs in stations.csv: {len(known_station_ids)}")

Gefundene Stationsdateien: 2675


Lese Stations-UUIDs:   0%|          | 0/2675 [00:00<?, ?it/s]

Anzahl eindeutiger UUIDs in stations.csv: 17802


In [6]:
# Abgleich: Sind die verdächtigen UUIDs in stations.csv bekannt?

suspect_df["known_in_stations_csv"] = suspect_df["station_uuid"].isin(known_station_ids)
orphan_df = suspect_df[~suspect_df["known_in_stations_csv"]].copy()

print(f"Verdächtige UUID/Sorte-Kombinationen gesamt:  {len(suspect_df)}")
print(f"Davon in stations.csv bekannt:                {suspect_df['known_in_stations_csv'].sum()}")
print(f"Davon NICHT bekannt (Orphans):                {len(orphan_df)}\n")

print("Alle verdächtigen Stationen:")
display(suspect_df)

if not orphan_df.empty:
    print("\nOrphans (nicht in stations.csv):")
    display(orphan_df)

suspect_df.to_csv("outlier_station_uuid_check.csv", index=False)
orphan_df.to_csv("outlier_station_uuid_orphans.csv", index=False)

Verdächtige UUID/Sorte-Kombinationen gesamt:  36
Davon in stations.csv bekannt:                36
Davon NICHT bekannt (Orphans):                0

Alle verdächtigen Stationen:


,fuel,station_uuid,outlier_count,known_in_stations_csv
0,diesel,15884845-a2be-4f77-a37b-1ac9e876a62a,137,True
1,diesel,8099817a-75c0-42a6-8b84-cbbf456f41bb,30,True
2,diesel,88b513b9-1369-460c-b878-ab33e6b3a177,7,True
3,diesel,6363e06a-ff31-4938-bbda-7dbda4a796b4,5,True
4,diesel,49164e7b-24bc-424d-7122-eed448b4b80c,3,True
5,diesel,62d75b56-be9c-45b6-a263-20df7a78d7c6,2,True
6,diesel,aa842438-c80d-46c1-828f-2cadb756d011,2,True
7,diesel,047f303e-8978-4106-990d-e62b8c45c0d8,1,True
8,diesel,0ce2fe47-449c-4cfa-843e-818cd482b624,1,True
9,diesel,748223cd-6cb2-4619-b8bf-c4987821d0ae,1,True
